(math)=
# A Brief Examination of Probability in Tu'urna

This appendix briefly explores the probability distributions associated with rolling dice using [Tu'urna's rules for dice rolling](gameplay:rolls). Unlike in many other game systems, such as Dungeons and Dragons, which use d20 dice in order to flatten out the roll distributions, Tu'urna uses a different system involving several six-sided dice. As a consequence of this system, {{skills}} and {{rolls}} of Tu'urna do not scale intuitively with one's {{skill}} {{level}}. Several examples of these consequences and explanations are provided here.

(math:technical)=
## Technical Notes

This notebook uses a Python library called `tuurna`, which is available on GitHub in the same repository as the rest of this website's contents: [noahbenson/tuurna](https://github.com/noahbenson/tuurna). The `tuurna` library itself uses the [icepool library](https://github.com/HighDiceRoller/icepool) to model the probabilities of individual rolls.

If you wish to reproduce the calculations on this website, you can use the following steps in a POSIX-compliant shell (e.g., BASH). Note that [Docker](https://docker.com/) must be installed and running and that the `$` characters represent the command prompt and are not part of the commands.

```bash
$ git clone https://github.com/noahbenson/tuurna
$ cd tuurna
$ docker compose up
```

This will start a [Jupyter server](https://jupyter-server.readthedocs.io/en/latest/) that contains an installation of the library as well as the repository of documentation. The Jupyter notebook containing the instructions for this web page can be found in the repository as the file [doc/math/dice.ipynb](https://github.com/noahbenson/tuurna/blob/main/doc/math/dice.ipynb). Jupyter code cells containing the instructions are hidden on this web page but are visible in the notebook file.

In [ ]:
# Dependencies 

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import icepool as ip

import tuurna as tu


# Utilities ##################################################################

def style_axes(ax, /, xticks=None, yticks=None, xlim=None, ylim=None, lw=1):
    """Styles the axes without modifying their limits or ticks.

    Stylized axes have no frames and plot the x and y axes as outward facing
    disconnected axes. Running this function before changing either the ticks
    or the limits will result in a mangled image.
    """
    # Process the options:
    if xticks is None:
        xticks = ax.get_xticks()
    else:
        ax.set_xticks(xticks)
    if yticks is None:
        yticks = ax.get_yticks()
    else:
        ax.set_yticks(yticks)
    if xlim is None:
        xlim = ax.get_xlim()
    else:
        ax.set_xlim(xlim)
    if ylim is None:
        ylim = ax.get_ylim()
    else:
        ax.set_ylim(ylim)
    # Clear the spines:
    for sp in ax.spines.values():
        sp.set_visible(False)
    # For each axis, figure out the min/max tick positions and the major
    # tick line width.
    ticklims = []
    tickopts = []
    for (axis, (mn,mx)) in zip((ax.xaxis, ax.yaxis), (xlim, ylim)):
        locs = np.concatenate(
            [axis.get_majorticklocs(), axis.get_minorticklocs()])
        locs = locs[(locs >= mn) & (locs <= mx)]
        ticklims.append((np.min(locs), np.max(locs)))
        try:
            ln = axis.get_majorticklines()[0]
        except Exception:
            opt = dict(lw=1.5, c='k', ls='-')
        else:
            opt = dict(lw=ln.get_lw(), c=ln.get_color(), ls=ln.get_linestyle())
        if opt['ls'] == 'None':
            del opt['ls']
        if lw is not None:
            opt['lw'] = lw
        tickopts.append(opt)
    # Draw the plot axis lines:
    (xopts, yopts) = tickopts
    ((xtickmin,xtickmax), (ytickmin,ytickmax)) = ticklims
    (xmin,xmax) = xlim
    (ymin,ymax) = ylim
    xln = ax.plot([xtickmin, xtickmax], [ymin, ymin], **xopts)
    yln = ax.plot([xmin, xmin], [ytickmin, ytickmax], **yopts)
    # Return these lines.
    return (xln, yln)


# Configuration ##############################################################

min_level = 0
max_level = 12
min_fate = -2
max_fate = 2
fate0 = 0

(math:checks)=
## How Likely is a Character to Succeed on a Check?

The most basic kind of {{roll}} made in Tu'urna is that of a {{skill}} {{check}}. {{Checks}} have three components: the {{skill}} {{level}} of the {{character}} performing the {{check}}, the {{difficulty}} of the {{check}}, and any {{modifiers}} ({{labor}} or {{leverage}}). Intuitively, the {{labor}} of a {{check}} can be added to the {{character}}'s {{level}}, since {{labor}} change the number of dice rolled; similarly, {{leverage}} may be subtracted from the {{difficulty}} since {{leverage}} changes the {{roll}}'s {{score}}. There are thus only two values to consider: the {{level}} and the {{difficulty}}.

The following image plots the probability of success ($y$-axis) for a {{character}} making a {{fate}}-neutral {{check}} in terms of the {{check}}'s {{difficulty}} ($x$-axis). The curves of different colors that are plotted represent different {{skill}} {{levels}} that the {{character}} can have.

In [ ]:
cm = mpl.cm.nipy_spectral

(fig,ax) = plt.subplots(1,1, figsize=(7,4.5), dpi=72*10)

for (ii,skill) in enumerate(range(1, 10)):
    roll = tu.skills.rolldist(skill)
    (x,y) = np.transpose(
        [(dl, float((roll >= dl).probability(True)))
         for dl in range(1, 11)])
    ax.plot(x, y, '.-', c=cm(ii / 9), label=f'{skill}')
for y in np.arange(0, 1.01, 0.1):
    ax.plot([0, 10], [y, y], c='0.5', lw=0.25, zorder=-10)
ax.set_xticks(range(1,11))
ax.set_xlabel('Difficulty – Leverage')
ax.set_xlim([0.6,11.4])
ax.set_ylabel('Probability of Success')
ax.set_ylim([-0.05, 1.19])
ax.legend(title="Level + Labor", loc='upper right')
style_axes(ax, lw=0.75)

plt.show()

```{note}

Because each die that is rolled during a {{check}} or {{contest}} can explode indefinitely (albeit with diminishing probability), a {{character}} with any {{skill}} {{level}} greater than 0 can theoretically {{roll}} any valid {{score}}. However, the probability of a {{score}} of 10 for a {{character}} with a {{level}} of 1, for example, is approximately 1 in 30,000.
```

The above plot shows the raw probability of success on a {{check}} in terms of {{skill}} {{level}} and {{difficulty}}, but these data are frequently more useful when organized according to the difference between one's {{level}} and the {{difficulty}}. Such an organization is useful for answering questions like, "what are a {{character}}'s chances of succeeding on a {{check}} whose {{difficulty}} is two greater than their {{skill}} {{level}}?"

The following image shows this arrangement of the data. The $x$-axis shows the {{character}}'s {{skill}} {{level}} while the $y$-axis shows the probability of success, just like in the plot above. However, here, the different curve separate the probabilities based on the {{difficulty}} of the {{check}} minus the {{character}}'s {{level}} in the associated {{skill}}. For example, the green curve labeled "0" plots the probability that a {{character}} succeeds on a {{check}} whose {{difficulty}} is equal to their {{skill}} {{level}}.

In [ ]:
cm = mpl.cm.nipy_spectral

(fig,ax) = plt.subplots(1,1, figsize=(7,4), dpi=72*10)

rolls = [
    (skill, tu.skills.rolldist(skill))
    for skill in range(1,10)]

gapmin = -4
gapmax = 4

for (ii,diff) in enumerate(range(gapmin, gapmax + 1)):
    (x,y) = np.transpose(
        [(skill, float((roll >= skill + diff).probability(True)))
         for (skill, roll) in rolls
         if skill + diff >= 0
         if skill + diff <= 10])
    ax.plot(x, y, '.-', c=cm(ii / (gapmax - gapmin)), label=f'{diff}')
for y in np.arange(0, 1.01, 0.1):
    ax.plot([0, 9], [y, y], c='0.5', lw=0.25, zorder=-10)
ax.set_xticks(range(1,10))
ax.set_xlabel('Skill Level')
ax.set_xlim([0.6,12.4])
ax.set_ylabel('Probability of Success')
ax.set_ylim([-0.05, 1.05])
ax.legend(title='Difficulty - Level', loc='right')
style_axes(ax, lw=0.75)

ax.set_title('Probability of Skill Check Success Based on Gap Between Difficulty and Level')
plt.show()

(math:contests)=
## Who is the Likely Winner of a Contest?

{{Contests}} are essentially {{checks}} where two {{characters}} pit their {{skills}} against each other. Whichever {{character}} {{rolls}} a higher {{score}} is the winner of a {{contest}}. Because the {{roll}} distributions are not uniform, these probabilities are somewhat unintuitive. The following image shows a matrix in which columns represent the {{level}} of one {{player_character}} ({{PC}}), the rows represent the {{level}} of the other {{PC}}, and the values in the cells represent the probability that {{PC}}1 defeats {{PC}}2 in a {{contest}}. For the purpose of this plot, there are not ties&mdash;if {{characters}} obtain tie {{scores}}, they repeat the {{contest}}.

In [ ]:
cmp_table = np.zeros((13,13))
for ii in range(13):
    r2 = tu.skills.rolldist(ii)
    for jj in range(13):
        r1 = tu.skills.rolldist(jj)
        p_eq = float((r1 == r2).probability(True))
        p_gt = float((r1 > r2).probability(True))
        if ii == jj:
            p = p_gt + 0.5*p_eq
        elif ii > jj:
            p = p_gt + p_eq
        else:
            p = p_gt
        cmp_table[ii,jj] = p

(fig,ax) = plt.subplots(1,1, figsize=(10,10))

ax.imshow(cmp_table, vmin=0, vmax=1, cmap='vanimo')
for ((j,i),label) in np.ndenumerate(cmp_table):
    c = str(round(1 - 2*abs(0.5 - label)))
    label = f'{int(round(label*100)):2d}'
    ax.text(i, j, label, c=c, ha='center', va='center', fontsize=10)

ax.invert_yaxis()
ax.set_xlabel('PC1 Level')
ax.set_ylabel('PC2 Level')
ax.set_title('Probability that PC1 Wins a Contest [%]')
ax.plot([9.5, 9.5], [-0.5, 12.5], 'w-', lw=1.5)
ax.plot([-0.5, 12.5], [9.5, 9.5], 'w-', lw=1.5)

plt.show()

Note that the above plot extends to {{level}} 12, even through most {{PCs}} can only achieve {{level}} 9 at most.

Unsurprisingly, the probability of one {{character}} defeating another when both have the same {{skill}} {{level}} is always 50%. As the {{levels}} of the contestants become different, however, the probabilities change quickly. The following image shows the same probabilities plotted above, but instead of organizing them as a matrix, it plots the probability of a lower-{{level}} {{character}} ("C1") defeating a higher-{{level}} {{character}} ("C2") in terms of the difference in their {{levels}}.

In [ ]:
cm = mpl.cm.nipy_spectral

(fig,ax) = plt.subplots(1,1, figsize=(7,4.5), dpi=720)

diffmax = 8
skillmax = 12
for skilldiff in range(diffmax + 1):
    d = np.diag(cmp_table, -skilldiff)
    mu = np.mean(
        [np.arange(skilldiff, skillmax + 1),
         np.arange(0, skillmax - skilldiff + 1)],
        axis=0)
    ax.plot(
        mu, d, '.-',
        c=cm(skilldiff / diffmax), 
        label=(r'L2 - L1 = ' + str(skilldiff)))
ax.set_xlim((-0.5, 14.6))
ax.set_xticks(range(0,13,1))
ax.set_ylim((-0.025,0.525))
ax.set_xlabel('Average Level of C1 and C2 (C1 < C2)')
ax.set_ylabel('Probability that C1 Wins')
ax.set_title('Probability that C1 defeats C2 by Level Difference')
ax.legend(title='C2 level - C1 level', loc='lower right')
style_axes(ax, lw=0.75)

plt.show()

(math:fate)=
## The Effects of a Roll's Fate

One other property of a {{roll}} not discussed above is the {{roll}}'s {{fate}}. The {{fate}} of a {{roll}} acts a bit like advantage and disadvantage in 5th edition Dungeons and Dragons in that it shifts the overall distribution of the value rolled. To visualize the effects of {{fate}}, we can start with a simple plot of the probability of success on a {{roll}} as a matrix where rows represent {{difficulties}} and columns represent {{skill}} {{levels}}. Below, we make a plot like this for each of the five {{fates}}: cursed, disadvantaged, neutral, advantaged, and blessed.

In [ ]:
im = np.array(
    [[[float((tu.skills.rolldist(lev, fate=fate) >= diff).probability(True))
       for diff in range(1,11)]
      for lev in range(1,13)]
     for fate in range(-2,3)])
dimnames = ['Fate', 'Level', 'Difficulty']

(fig,axs) = plt.subplots(5,1, figsize=(5,18), dpi=288) #, sharex=True, sharey=True)
fig.subplots_adjust(0,0,1,1,0,0.25)

for (fval, sl, ax) in zip(range(-2,3), im, axs.flat):
    sl = sl.T
    plotim = np.ones_like(sl)
    plotim[sl < 0.99] = 0.9
    plotim[sl < 0.80] = 0.65
    plotim[sl < 0.50] = 0.3
    plotim[sl < 0.05] = 0
    (r,c) = plotim.shape
    ax.imshow(
        plotim, 
        extent=(0.5, c+0.5, r+0.5, 0.5),
        vmin=0,
        vmax=1,
        cmap='magma')
    ax.invert_yaxis()
    for ((j,i),label) in np.ndenumerate(sl):
        label = f'{int(round(label*100)):2d}'
        ax.text(i+1, j+1, label, c='b', ha='center', va='center', fontsize=8)
    ax.set_title(tu.skills.to_fate(fval).name.capitalize())
if len(axs.shape) < 2:
    axs = axs[:,None]
for ax in axs.flat:
    ax.set_ylabel('Difficulty')
    ax.set_xlabel('Level')
    ax.set_xticks(range(1,13))
    ax.set_yticks(range(1,11))

pass